# Reproduction Code Notebook
This notebook contains all the code needed to reproduce the results in our JOS submission.

In [56]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from modules.utils import (
    load_ateco_data,
    load_query_embeddings
)
from modules.semantic import (
    get_knowledge_base,
    get_similarity,
    extract_top_k,
    get_matches,
    print_results
)
from modules import plots

## Configs
Here, you can set some configuration parameters for the different models and approaches.

In [57]:
APPROACH : str  = "naive"  # "naive" | "disentangled" | "centroid"
MODEL_ID : str  = "paraphrase-multilingual-MiniLM-L12-v2"
SYNTHETIC: bool = False

---

## Pre-processing
Load the ateco data, whether the descriptors, official disentangled classification or synthetic queries, along with the dataset containing CIRCE labels and word counts.

In [58]:
ateco_df = load_ateco_data(approach=APPROACH, synthetic=SYNTHETIC)
circe_df = pd.read_csv("data/ateco22_circe_labels.csv")

circe_labels = circe_df["code"].tolist()

Load the sentence-level embedding model.

In [59]:
model = SentenceTransformer(MODEL_ID, trust_remote_code=True)

Encode the texts using the embedding model keeping track of the codes.

In [ ]:
codes, base_embeddings = get_knowledge_base(model, ateco_df, APPROACH)

Load the CIRCE pre-computed query embeddings along with the CIRCE-assigned labels.

In [61]:
query_embeddings = load_query_embeddings(model_id=MODEL_ID)
circe_labels = pd.read_csv("data/ateco22_circe_labels.csv")["code"].tolist()

---

## Evaluation
Compute the similarity between each query embedding and the vectors in the knowledge base.
> For the `"disentangled"` approach, set `top_k` to a higher number, e.g. 20-30, so that you are sure to find at least `max(ks)` unique codes in the extracted results.

In [62]:
sims, ids = get_similarity(query_embeddings, base_embeddings, top_k=5)

Extract the matching at the category (5-digit) and division (2-digit) levels.

In [ ]:
perf_cat, perf_div = get_matches(sims, ids, codes, circe_labels, APPROACH, ks=[1, 3, 5])

print_results(perf_cat, perf_div)

### Matching by division
Compute the division-specific matching between semantic and CIRCE approaches.

In [ ]:
match_at_5 = perf_div[5]

df_viz = pd.DataFrame({"code": circe_labels, "is_match": match_at_5})
df_viz["division"] = df_viz["code"].str[:2]

df_viz = df_viz.groupby("division").agg({"is_match": "mean"})

plots.plot_division_match(df_viz.index, df_viz["is_match"])

### Similarity thresholds
Inspect the effect of setting different similarity thresholds on matching performance.

In [ ]:
similarity_thresholds = np.arange(0.5, 0.91, 0.05)

match_at_1 = perf_cat[1]
top_sim = [s[0].item() for s in sims]

df_viz = pd.DataFrame({"is_match": match_at_1, "sim": top_sim})

classified, matching = [], []
for s_t in similarity_thresholds:
    filtered_df = df_viz[df_viz["sim"] >= s_t]
    _classified = len(filtered_df)
    _matching = filtered_df["is_match"].mean()

    classified.append(_classified)
    matching.append(_matching)

plots.plot_thresholds(similarity_thresholds, classified, matching)

### Semantic richness and performance
Inspect the effect of semantically rich queries on matching performance.

In [ ]:
k = 1
min_words = 5

n_words = circe_df["n_words"].tolist()

df_raw = pd.DataFrame({f"match@{k}": perf_cat[k], f"match@{k}_div": perf_div[k], "n_words": n_words})
df_rich = df_raw[df_raw["n_words"] >= min_words]

print(f"Matching on queries with more than {min_words-1} words:")
print(f"* Total queries: {len(df_raw)} -> {len(df_rich)}")
print(f"* Match@{k} (2-digit): {df_raw[f'match@{k}_div'].mean():.4f} -> {df_rich[f'match@{k}_div'].mean():.4f}")
print(f"* Match@{k} (5-digit): {df_raw[f'match@{k}'].mean():.4f} -> {df_rich[f'match@{k}'].mean():.4f}")